# Step 1: Load the Dataset

In this step, we load the job dataset into our notebook using the pandas library.

- Pandas helps us work with data in table format (like Excel).
- We store the dataset in a variable called `job_data`.
- Then we display the first few rows to understand what the data looks like. 

In [2]:
import pandas as pd

# Load dataset from the data folder
job_data = pd.read_csv("../data/raw/jobs.csv")

# Display first 5 rows
job_data.head()

,job_id,category,job_title,job_description,job_skill_set
0,3902668440,HR,Sr Human Resource Generalist,SUMMARY\nTHE SR. HR GENERALIST PROVIDES HR EXP...,"['employee relations', 'talent acquisition', '..."
1,3905823748,HR,Human Resources Manager,BE PART OF A STELLAR TEAM AT YSB AS THE MANAGE...,"['Talent Acquisition', 'Employee Performance M..."
2,3905854799,HR,Director of Human Resources,OUR CLIENT IS A THRIVING ORGANIZATION OFFERING...,"['Human Resources Management', 'Recruitment', ..."
3,3905834061,HR,Chief Human Resources Officer,JOB TITLE: CHIEF HUMAN RESOURCES OFFICER (CHRO...,"['talent management', 'organizational developm..."
4,3906250451,HR,Human Resources Generalist (Hybrid Role),DESCRIPTION\n\n WHO WE ARE \n\nAVI-SPL IS A DI...,"['Microsoft Office', 'Data analysis', 'Employe..."


# Step 2: Basic Data Exploration

In this step, we check the structure of the dataset.

- View column names
- Check data types
- Identify missing values

This helps us understand what data we are working with before cleaning or modeling.

In [3]:
# Show column names
job_data.columns

Index(['job_id', 'category', 'job_title', 'job_description', 'job_skill_set'], dtype='str')

# Step 3: Select Relevant Column

From the dataset, we select the `job_description` column.

Reason:
- It contains detailed information about each job
- It is suitable for text-based matching using TF-IDF

We ignore other columns for now to keep the model simple.

In [ ]:
# Keep only the job_description column
job_data = job_data[['job_description']]

# Remove missing values (if any)
job_data = job_data.dropna()

# Display again
job_data.head()

# Step 4: Text Preprocessing

In this step, we clean the job descriptions before applying TF-IDF.

Why cleaning is needed:
- Remove unnecessary symbols and noise
- Standardize text (convert to lowercase)
- Make the data consistent for better matching

This helps improve the accuracy of similarity calculations.

In [6]:
import re

def clean_text(text):
    text = text.lower()
    
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    
    # Replace newline with space
    text = re.sub(r'\n', ' ', text)
    
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply cleaning function
job_data['cleaned_description'] = job_data['job_description'].apply(clean_text)

# Show results
job_data[['job_description', 'cleaned_description']].head()

,job_description,cleaned_description
0,SUMMARY\nTHE SR. HR GENERALIST PROVIDES HR EXP...,summary the sr hr generalist provides hr exper...
1,BE PART OF A STELLAR TEAM AT YSB AS THE MANAGE...,be part of a stellar team at ysb as the manage...
2,OUR CLIENT IS A THRIVING ORGANIZATION OFFERING...,our client is a thriving organization offering...
3,JOB TITLE: CHIEF HUMAN RESOURCES OFFICER (CHRO...,job title chief human resources officer chroin...
4,DESCRIPTION\n\n WHO WE ARE \n\nAVI-SPL IS A DI...,description who we are avispl is a digital ena...


# Step 5: Convert Text into Numerical Vectors (TF-IDF)

In this step, we convert the cleaned job descriptions into numerical vectors.

Why this is needed:
- Machines cannot understand text directly
- TF-IDF converts words into numbers based on importance

What happens here:
- Each job description becomes a vector
- These vectors will be used to compare similarity with user input

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=10000,
    min_df=8,              # ignore very rare words
    max_df=0.8,            # ignore overly common words
    token_pattern=r'(?u)\b[a-zA-Z]{3,}\b'  # keep words with ≥3 letters only
)

# Fit and transform job descriptions
job_vectors = vectorizer.fit_transform(job_data['cleaned_description'])

# Check shape of the matrix
job_vectors.shape

(1167, 3814)

In [18]:
feature_names = vectorizer.get_feature_names_out()

print(feature_names[:200])# first 50 words

['abide' 'abilities' 'ability' 'able' 'abreast' 'absence' 'academic'
 'accelerate' 'accept' 'acceptable' 'acceptance' 'accepted' 'accepting'
 'access' 'accessibility' 'accessible' 'accessories' 'accident'
 'accommodate' 'accommodation' 'accommodations' 'accomplish'
 'accomplished' 'accordance' 'according' 'accordingly' 'account'
 'accountabilities' 'accountability' 'accountable' 'accountant'
 'accounting' 'accountingfinance' 'accounts' 'accredited' 'accruals'
 'accuracy' 'accurate' 'accurately' 'achieve' 'achieved' 'achievement'
 'achievements' 'achieves' 'achieving' 'acquire' 'acquired' 'acquiring'
 'acquisition' 'acquisitions' 'act' 'acting' 'action' 'actionable'
 'actions' 'active' 'actively' 'activities' 'activity' 'acts' 'actual'
 'acumen' 'ada' 'adapt' 'adaptability' 'adaptable' 'add' 'added' 'adding'
 'addition' 'additional' 'additionally' 'address' 'addressed' 'addresses'
 'addressing' 'adept' 'adequate' 'adhere' 'adherence' 'adheres' 'adhering'
 'adhoc' 'adjust' 'adjusted' 'ad

In [19]:
import numpy as np

vector = job_vectors[0].toarray()[0]
non_zero_indices = np.where(vector > 0)[0]

for i in non_zero_indices[:20]:
    print(feature_names[i], ":", vector[i])

ability : 0.10342725922175693
accordance : 0.045626632617725274
accuracy : 0.04292248750560695
achieve : 0.0342498691606137
acquisition : 0.04363845803064645
actionable : 0.054912207037197534
actions : 0.04621362314620692
ada : 0.05718027078163982
administering : 0.1181780253253317
administration : 0.12776877991517155
administrative : 0.04224455149043043
align : 0.13319095737491918
analytical : 0.034668424516266956
andor : 0.026898113114187418
applicable : 0.033507188446561524
areas : 0.06803343148815685
aspects : 0.0809977034772955
assessments : 0.05683052850191638
assigned : 0.14488262496019122
assist : 0.06921556809497434


# Step 6: Compute Skill Match Score using Cosine Similarity

In this step, we compare the user’s skills with all job descriptions.

Process:
1. Convert user input into a TF-IDF vector
2. Compare it with all job vectors
3. Compute similarity scores using cosine similarity

Output:
- A score for each job (higher = better match)
- Top matching jobs are selected

In [21]:
from sklearn.metrics.pairwise import cosine_similarity

# User input (skills)
user_input = "HR Communication Employee Relations"

# Convert user input into vector
user_vector = vectorizer.transform([user_input])

# Compute similarity with all job descriptions
similarities = cosine_similarity(user_vector, job_vectors)

# Convert to 1D array
scores = similarities.flatten()

# Get top 5 matching jobs
top_indices = scores.argsort()[-5:][::-1]

# Display results
print("Top 5 Job Matches:\n")

for i in top_indices:
    print("Score:", round(scores[i], 4))
    print(job_data.iloc[i]['job_description'][:200])
    print("-" * 50)

Top 5 Job Matches:

Score: 0.3206
HR GENERALISTWE ARE SEEKING AN EXPERIENCED HR GENERALIST TO JOIN OUR TEAM AND SUPPORT VARIOUS HUMAN RESOURCES FUNCTIONS. THE IDEAL CANDIDATE WILL HAVE A COMPREHENSIVE UNDERSTANDING OF HR PRINCIPLES AN
--------------------------------------------------
Score: 0.3158
TALENT RECRUITING PARTNERS IS SEEKING A SKILLED DIRECTOR OF HUMAN RESOURCES / OPERATIONS LEADER TO OVERSEE ALL EMPLOYEE RELATIONS AS THE LIAISON BETWEEN OUR CLIENT’S CORPORATE OFFICE AND THIS LOCATION
--------------------------------------------------
Score: 0.2995
POSITION OVERVIEWWE ARE SEEKING AN EXPERIENCED HUMAN RESOURCES (HR) MANAGER TO JOIN OUR CLIENT'S TEAM. AS THE HR MANAGER, YOU WILL PLAY A PIVOTAL ROLE IN SHAPING THE ORGANIZATION’S SUCCESS BY OVERSEEI
--------------------------------------------------
Score: 0.2919
DIRECTOR OF HUMAN RESOURCES - ET GLOBAL  RESPONSIBLE FOR OVERSEEING ALL ASPECTS OF ALL HR MANAGEMENT WITHIN THE COMPANY. THIS INCLUDES RECRUITMENT, EMPLOYEE RELATIONS, 

# Step 7: Display Job Title and Matched Skills

In this step, we improve the output by showing:
- Job Title
- Similarity Score
- Matched Skills (common keywords)

Matched skills are identified by finding overlapping important words
between user input and job description.

In [27]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

user_input = "talent acquisition recruitment sourcing interviewing candidate screening employer branding applicant tracking system ats communication negotiation"

user_vector = vectorizer.transform([user_input])
scores = cosine_similarity(user_vector, job_vectors).flatten()
top_indices = scores.argsort()[-5:][::-1]

user_words = set(user_input.lower().split())

print("Top 5 Job Matches:\n")

for i in top_indices:
    job_title = job_data.iloc[i].get('job_title', 'N/A')
    job_desc = job_data.iloc[i]['cleaned_description']
    job_vector = job_vectors[i].toarray()[0]

    # indices of non-zero features
    nz = np.where(job_vector > 0)[0]

    # sort by TF-IDF weight (descending)
    sorted_idx = nz[np.argsort(job_vector[nz])[::-1]]

    # --- Top keywords (filter out very weak ones) ---
    top_keywords = []
    for j in sorted_idx:
        if job_vector[j] < 0.05:   # threshold (tune if needed)
            continue
        word = feature_names[j]
        top_keywords.append(word)
        if len(top_keywords) == 5:
            break

    # --- Matched skills (only keep if also important in this job) ---
    job_words = set(job_desc.split())
    matched = []
    for w in user_words:
        if w in job_words:
            # keep only if it has some TF-IDF weight
            if w in feature_names:
                idx = np.where(feature_names == w)[0]
                if len(idx) > 0 and job_vector[idx[0]] > 0.03:
                    matched.append(w)

    print("Score:", round(scores[i], 4))
    print("Job Title:", job_title)
    print("Matched Skills:", matched)
    print("Top Keywords:", top_keywords)
    print("-" * 50)

Top 5 Job Matches:

Score: 0.2517
Job Title: Human Resources Manager
Matched Skills: ['screening', 'applicant', 'sourcing', 'negotiation', 'talent', 'candidate', 'acquisition', 'interviewing', 'recruitment', 'tracking']
Top Keywords: ['human', 'supervisors', 'talent', 'practices', 'recruitment']
--------------------------------------------------
Score: 0.2041
Job Title: Human Resources Generalist
Matched Skills: ['screening', 'employer', 'sourcing', 'talent', 'interviewing', 'recruitment']
Top Keywords: ['employee', 'recruitment', 'interviewing', 'screening', 'hrrelated']
--------------------------------------------------
Score: 0.1642
Job Title: Human Resources Recruiter
Matched Skills: ['screening', 'candidate', 'interviewing', 'recruitment', 'ats']
Top Keywords: ['interviews', 'hiring', 'recruiting', 'screening', 'making']
--------------------------------------------------
Score: 0.1591
Job Title: Senior Human Resources Generalist
Matched Skills: ['talent', 'candidate', 'acquisition

# Step 8: Skill-Based Score Calculation

In this step, we calculate a skill match score by comparing
user skills with job required skills.

This score is combined with TF-IDF similarity to produce a final score.

In [ ]:
# Step 8.1 — Calculate skill overlap score
def skill_match_score(user_skills, job_skills):
    matched = 0

    for skill in job_skills:
        cleaned_skill = skill.lower()

        for user_skill in user_skills:
            if user_skill in cleaned_skill or cleaned_skill in user_skill:
                matched += 1
                break

    if len(job_skills) == 0:
        return 0

    return matched / len(job_skills)

: 